# About

This notebook is for testing if the dataset cleaning was proper or not, as I encountered that a significant amount of data was removed from the dataset after cleaning.

## Imports

In [1]:
import pandas as pd
from src.core.config import load_validation_schema, load_validation_rules, load_cleaning_configuration
from src.preprocessing.preprocessing import PreprocessingPipeline, initial_cleanup
from src.dataset.loader import load_cicids2017_dataset
from pathlib import Path

## Loading Dataset

In [2]:
dataset_dir = '../dataset'

df = load_cicids2017_dataset(dataset_dir)

In [3]:
df = initial_cleanup(df)

In [4]:
df.drop(columns=["Fwd Header Length.1"], inplace=True)

# doing the above because the column is redundant.

## Validation & Cleaning

In [5]:
sp = Path('../config/preprocessing/validation_shema.yaml')
schema_cfg = load_validation_schema(sp)

rp = Path('../config/preprocessing/validation_rules.yaml')
rules_cfg = load_validation_rules(rp)

cleaning_path = Path('../config/preprocessing/cleaning.yaml')
cleaning_cfg = load_cleaning_configuration(cleaning_path)

preprocess = PreprocessingPipeline(schema_cfg, rules_cfg, cleaning_cfg)

preprocess.validate(df)
df_cleaned = preprocess.clean(df)

preprocess.validate(df_cleaned)
df_cleaned_final = preprocess.clean(df_cleaned)

preprocess.validate(df_cleaned_final)

print(preprocess.validation_result[f'validation_result_{len(preprocess.validation_result)}'])

Validation Result
----------------------
Repairable: 0
Non-repairable: 0
NaN*: 0
Inf*: 0
Negative*: 0
Out Of range*: 0
Rule violators: {'R001': np.int64(0), 'R002': np.int64(0), 'R003': np.int64(0), 'R004': np.int64(0), 'R005': np.int64(0), 'R006': np.int64(0), 'R007': np.int64(0), 'R008': np.int64(0), 'R009': np.int64(0), 'R010': np.int64(0), 'R011': np.int64(0), 'R012': np.int64(0), 'R013': np.int64(0), 'R014': np.int64(0), 'R015': np.int64(0), 'R016': np.int64(0), 'R017': np.int64(0), 'R018': np.int64(0), 'R019': np.int64(0)}
----------------------
NOTE: those marked with * are showing individual cell value count, not row.
Just so you know, a single row can have multiple NaN, Inf, Negative etc. values.


## Comparison

In [6]:
df_comparison = pd.DataFrame({
    'Original': [],
    'First Clean': [],
    'First Remove': [],
    'First remove percent': [],
    'Second Remove': [],
    'Second Removed': [],
    'Second Removed Percent': []
})

df_comparison['Original'] = df['Label'].value_counts()
df_comparison['First Clean'] = df_cleaned['Label'].value_counts()
df_comparison['First Remove'] = df['Label'].value_counts() - df_cleaned['Label'].value_counts()
df_comparison['First remove percent'] = 100 -(df_cleaned['Label'].value_counts() / df['Label'].value_counts()) * 100
df_comparison['Second Clean'] = df_cleaned_final['Label'].value_counts()
df_comparison['Second Remove'] = df_cleaned['Label'].value_counts() - df_cleaned_final['Label'].value_counts()
df_comparison['Second remove percent'] =  100 - (df_cleaned_final['Label'].value_counts() / df_cleaned['Label'].value_counts()) * 100

In [7]:
comp_path = Path('../out/data/')
comp_path.mkdir(parents=True, exist_ok=True)
df_comparison.to_csv(Path(comp_path, 'df_comparison.csv'))

In [8]:
df_comparison

,Original,First Clean,First Remove,First remove percent,Second Remove,Second Removed,Second Removed Percent,Second Clean,Second remove percent
Label,,,,,,,,,
BENIGN,2096484,531131.0,1565353.0,74.665631,0.0,NaN,NaN,531131.0,0.0
DoS Hulk,172849,8662.0,164187.0,94.988690,0.0,NaN,NaN,8662.0,0.0
DDoS,128016,93.0,127923.0,99.927353,0.0,NaN,NaN,93.0,0.0
PortScan,90819,90408.0,411.0,0.452548,0.0,NaN,NaN,90408.0,0.0
DoS GoldenEye,10286,81.0,10205.0,99.212522,0.0,NaN,NaN,81.0,0.0
FTP-Patator,5933,5864.0,69.0,1.162987,0.0,NaN,NaN,5864.0,0.0
DoS slowloris,5385,3177.0,2208.0,41.002786,0.0,NaN,NaN,3177.0,0.0
DoS Slowhttptest,5228,1123.0,4105.0,78.519510,0.0,NaN,NaN,1123.0,0.0
SSH-Patator,3219,3187.0,32.0,0.994098,0.0,NaN,NaN,3187.0,0.0


From the above I can say for certain that the cleaning  was either wrong or too aggressive and removed a lot of data.